In [2]:
# Paramètres
import io
import pandas as pd
import boto3
import io
import unicodedata
import re
import numpy as np
import unicodedata
import unicodedata, re
import re

## Chargement de la base panel_solde_df_2  (Poste )

In [3]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"
object_key = "Solde/panel_solde_df_2.parquet"

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Afficher un aperçu
panel_solde_df.head()

,solde de base,salaire d agent temporaire,salaire de gens de maison,forfait indiciaire ambassadeur,compensatrice smig,revalorisation 2014,compensatrice non imp ambassades,risque sanitaire,compensatrice non imposable,compensatrice imposable,...,judicature secret greffes,precompte synaps ci (syndicat national des professionnels de service social de cote d ivoire),ordre de recette divers,regularisation is,indemnite de logement (code 265),contribution nationale de solidarite,somme a recuperer par ordre de recette,remboursement caisse d avance,MONTANT BRUT,MATRICULE||CODE_ORGANISME
0,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,34560,011242X15
1,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,42400,012856Q15
2,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,45600,013454N15
3,None,None,None,None,None,None,None,None,None,62380,...,None,None,None,None,None,None,None,None,800000,027861L25
4,513605,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,842646,047690HBQ


# Fonctions nécessaires

## Affiche les Bases répétées détectée dans la colonne 

In [4]:
import re
import pandas as pd
from typing import Dict, List, Iterable

def _base_name(col: str) -> str:
    """
    Normalise un nom de colonne pour détecter les répétitions logiques.
    - retire un suffixe final entre parenthèses (ex: '(code 263)', '(A)', '(v2)')
    - remplace séparateurs (_ - . | /) par un espace
    - compacte les espaces et met en minuscules
    """
    s = str(col).strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)             # retire '(...)' en fin
    s = re.sub(r"[_\-|./]+", " ", s)                   # uniformise séparateurs
    s = re.sub(r"\s+", " ", s).strip()                 # compacte espaces
    return s.lower()

def find_repeated_columns(df: pd.DataFrame, exclude: Iterable[str] = ("SOURCE_FICHIER",)) -> Dict[str, List[str]]:
    """
    Regroupe les colonnes par 'base' et retourne les bases qui ont > 1 colonne.
    """
    groups: Dict[str, List[str]] = {}
    for c in df.columns:
        if c in exclude:
            continue
        base = _base_name(c)
        groups.setdefault(base, []).append(c)
    return {base: cols for base, cols in groups.items() if len(cols) > 1}

def report_repeated_columns(df: pd.DataFrame, exclude: Iterable[str] = ("SOURCE_FICHIER",)) -> pd.DataFrame:
    """
    Affiche un résumé lisible et renvoie un DataFrame des bases répétées.
    """
    rep = find_repeated_columns(df, exclude=exclude)
    if not rep:
        print("✅ Aucune colonne répétée détectée (selon la règle de normalisation).")
        return pd.DataFrame(columns=["base", "nb_colonnes", "colonnes"])

    print(f"🔎 Bases répétées détectées: {len(rep)}")
    rows = []
    for base, cols in sorted(rep.items()):
        print(f"  • {base}  ->  {cols}")
        rows.append({"base": base, "nb_colonnes": len(cols), "colonnes": cols})

    return (pd.DataFrame(rows)
              .sort_values(["nb_colonnes", "base"], ascending=[False, True])
              .reset_index(drop=True))


## Fonction Python de consolidation des colonnes répétées par année détectée

In [5]:
import re
import numpy as np
import pandas as pd
from typing import Optional, Iterable, Dict, List

# --- Année depuis SOURCE_FICHIER : gère MMYYYY et YYYY ---
def year_from_source(text: str) -> Optional[int]:
    s = str(text)
    m = re.search(r'(?<!\d)(0[1-9]|1[0-2])((?:19|20)\d{2})(?!\d)', s)
    if m:
        return int(m.group(2))
    m = re.search(r'(?:19|20)\d{2}', s)
    if m:
        return int(m.group(0))
    return None

def extract_years_series(src: pd.Series) -> pd.Series:
    return src.map(year_from_source)

def extract_years_unique(src: pd.Series) -> list[int]:
    years = extract_years_series(src).dropna().astype(int).unique().tolist()
    years.sort()
    return years

# --- Normalisation de "base" de colonne pour trouver les répétitions ---

def find_repeated_columns(df: pd.DataFrame, exclude: Iterable[str] = ("SOURCE_FICHIER",)) -> Dict[str, List[str]]:
    groups: Dict[str, List[str]] = {}
    for c in df.columns:
        if c in exclude:
            continue
        base = _base_name(c)
        groups.setdefault(base, []).append(c)
    return {base: cols for base, cols in groups.items() if len(cols) > 1}

# --- Détection "rempli" (vectorisée, colonne par colonne, pas par ligne) ---
def _filled_mask(block: pd.DataFrame, treat_zero_as_filled: bool) -> np.ndarray:
    """
    Retourne un tableau bool (n_rows, n_cols) indiquant si chaque cellule est "renseignée".
    - numériques : non-NaN (et !=0 si treat_zero_as_filled=False)
    - non-numériques : non-NaN et non vide après strip/lower et pas dans {'na','nan','none'}
    """
    n, m = block.shape
    filled = np.zeros((n, m), dtype=bool)
    # traiter colonne par colonne (m est petit : 2-10 dans tes cas)
    for j, col in enumerate(block.columns):
        s = block[col]
        if pd.api.types.is_numeric_dtype(s):
            ok = s.notna().to_numpy()
            if not treat_zero_as_filled:
                v = s.to_numpy()
                ok &= np.asarray(v != 0)
        else:
            # on évite astype("string") massif; on reste en object et on nettoie
            v = s.astype("object").to_numpy()
            # notna
            ok = pd.notna(v)
            # convertir en str seulement là où ok==True
            tmp = np.empty_like(v, dtype=object)
            tmp[ok] = np.char.lower(np.char.strip(v[ok].astype(str)))
            # vides ou placeholders
            bad = np.zeros(ok.shape, dtype=bool)
            if ok.any():
                vv = tmp[ok]
                bad_ok = (vv == "") | (vv == "na") | (vv == "nan") | (vv == "none")
                bad[ok] = bad_ok
            ok &= ~bad
        filled[:, j] = ok
    return filled  # (n_rows, n_cols)

# --- Consolidation rapide = "first_non_null" (coalesce) ---
def _coalesce_first_non_null(block: pd.DataFrame, treat_zero_as_filled: bool) -> pd.Series:
    """
    Retourne la première valeur renseignée par ligne, vectorisé via argmax.
    """
    if block.shape[1] == 1:
        s = block.iloc[:, 0]
        if pd.api.types.is_numeric_dtype(s) and not treat_zero_as_filled:
            return s.where(s.notna() & (s != 0))
        return s

    filled = _filled_mask(block, treat_zero_as_filled)  # (n, m)
    any_filled = filled.any(axis=1)
    # indices de la première True par ligne ; si aucune, 0 par défaut (on corrigera ensuite)
    first_idx = filled.argmax(axis=1)
    arr = block.to_numpy(copy=False)
    # sélectionner arr[row, first_idx[row]]
    rows = np.arange(arr.shape[0])
    out_vals = np.empty(arr.shape[0], dtype=object)
    out_vals[:] = pd.NA
    out_vals[any_filled] = arr[rows[any_filled], first_idx[any_filled]]
    return pd.Series(out_vals, index=block.index, dtype="object")

# --- Fonction principale optimisée (sans concat, suppression des sources) ---
def consolidate_all_years_first_non_null_and_drop_sources(
    df: pd.DataFrame,
    source_col: str = "SOURCE_FICHIER",
    *,
    treat_zero_as_filled: bool = True,
    new_name_fmt: str = "{base}"
) -> pd.DataFrame:
    """
    Consolidation haute performance :
      - détecte les bases répétées,
      - consolide par année (YYYY dans SOURCE_FICHIER, incl. MMYYYY.xlsx) en prenant la 1ère valeur renseignée,
      - supprime systématiquement les colonnes sources.
    """
    if source_col not in df.columns:
        raise ValueError(f"Colonne {source_col!r} absente.")

    out = df.copy()
    repeated = find_repeated_columns(out, exclude=(source_col,))
    if not repeated:
        print("ℹ️ Aucune base répétée. Rien à faire.")
        return out

    years = extract_years_unique(out[source_col])
    if years:
        out["_YEAR_"] = extract_years_series(out[source_col])
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            if new_col not in out.columns:
                out[new_col] = pd.NA
            for y in years:
                mask = out["_YEAR_"].eq(y)
                if not mask.any():
                    continue
                block = out.loc[mask, cols]
                out.loc[mask, new_col] = _coalesce_first_non_null(block, treat_zero_as_filled)
        out.drop(columns=["_YEAR_"], inplace=True, errors="ignore")
    else:
        # fallback: pas d'année -> coalesce sur toutes les lignes
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            block = out[cols]
            out[new_col] = _coalesce_first_non_null(block, treat_zero_as_filled)

    # drop colonnes sources
    cols_to_drop = sorted({c for cols in repeated.values() for c in cols})
    out.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    return out


## Affichage des colonnes qui se repètent

In [6]:
# Exemple
resume = report_repeated_columns(panel_solde_df)
# Si tu veux juste savoir s'il y en a :
has_repeats = not find_repeated_columns(panel_solde_df) == {}
print("Colonnes répétées ?", has_repeats)


🔎 Bases répétées détectées: 4
  • differentiel familial  ->  ['differentiel familial (code 221)', 'differentiel familial (code 422)']
  • indemnite de logement  ->  ['indemnite de logement (code 263)', 'indemnite de logement (code 271)', 'indemnite de logement (code 265)']
  • participation judicature  ->  ['participation judicature (code 440)', 'participation judicature (code 441)', 'participation judicature (code 442)', 'participation judicature (code 443)']
  • responsabilite tresor  ->  ['responsabilite tresor (code 487)', 'responsabilite tresor (code 493)']
Colonnes répétées ? True


## Consolidation des colonnes répétées par année détectée 

In [7]:
panel_solde_df = consolidate_all_years_first_non_null_and_drop_sources(
    panel_solde_df,
    source_col="SOURCE_FICHIER",
    treat_zero_as_filled=True,  # mets False si "0" doit compter comme vide
    new_name_fmt="{base}"       # ex. 'indemnite de logement', 'participation judicature', etc.
)



## S’assurer qu’il ne reste plus de colonnes suffixées "(code ...)"

In [8]:
# => doit renvoyer []
[c for c in panel_solde_df.columns if "(code" in c.lower()]

[]

## Chargement de la base panel_solde_df_1

In [9]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"
object_key = "Solde/panel_solde_df_1_code.parquet"

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df_1 = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Afficher un aperçu
panel_solde_df_1.head()

,MATRICULE||CODE_ORGANISME,MATRICULE,NOM,DATE NAISSANCE,SEXE,SITUATION MATRIMONIALE,NOMBRE ENFANT,LIEU AFFECTATION,SERVICE,ORGANISME,...,SERVICE_NORM,CODE_SERVICE,ORGANISME_NORM,CODE_ORGANISME,CLASSE/ECHELON_NORM,CODE_CLASSE/ECHELON,EMPLOI_NORM,CODE_EMPLOI,SITUATION_NORM,CODE_SITUATION
0,011242X15,011242X,DOSSO MEGBO,1939-01-01,Masculin,Marié,0.0,Seguela,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
1,012856Q15,012856Q,KHISSY BEYNIOUAH FULBERT,1939-01-01,Masculin,Célibataire,0.0,Bouaké,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
2,013454N15,013454N,VAHA TOMAS GNOMBLEI HENRI,1924-01-01,Masculin,Marié,0.0,Guiglo,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
3,027861L25,027861L,CAPET YATO MATHIEU,1943-03-01,Masculin,Marié,0.0,Abidjan,A affecter,"Min. Affaires Etrangères, de l'Intégration Afr...",...,a affecter,1800,"min affaires etrangeres, de l integration afri...",25,,<NA>,,<NA>,en activite,10
4,039659M38,039659M,COULIBALY YOUSSOUF,1945-12-01,Masculin,Marié,2.0,Abidjan,Dir Personnel Police Nationale,Min. de l'Intérieur et de la Sécurité,...,dir personnel police nationale,3813,min de l interieur et de la securite,38,commissaire divis 1er ech,LW,commissaire de police,P1ZC,retraite pour limite age,60


## Fusion des deux bases 

In [1]:
# --- 0) Copies de travail
dfL = panel_solde_df.copy()      # LEFT  : ta base 2 (montants, etc.)
dfR = panel_solde_df_1.copy()    # RIGHT : ta base 1 (état civil, codes, etc.)

# --- 1) Construire MATRICULE dans dfL à partir de 'MATRICULE||CODE_ORGANISME'
if "MATRICULE" not in dfL.columns:
    if "MATRICULE||CODE_ORGANISME" not in dfL.columns:
        raise KeyError("dfL doit contenir 'MATRICULE' ou 'MATRICULE||CODE_ORGANISME'")
    # prendre la partie avant '||' puis extraire un bloc alphanumérique
    raw = dfL["MATRICULE||CODE_ORGANISME"].astype("string")
    left_part = raw.str.split("||", n=1, expand=True).iloc[:, 0]
    dfL["MATRICULE"] = left_part.str.extract(r"([A-Za-z0-9]+)", expand=False)

# --- 2) Normaliser les clés (types + trim)
for d in (dfL, dfR):
    # SOURCE_FICHIER
    d["SOURCE_FICHIER"] = (d["SOURCE_FICHIER"].astype("string")
                                            .str.strip()
                                            .str.replace("\u00A0","", regex=False))
    # MATRICULE (présent des deux côtés maintenant)
    d["MATRICULE"] = (d["MATRICULE"].astype("string")
                                    .str.strip()
                                    .str.replace("\u00A0","", regex=False))
    # Option : homogénéiser la casse si nécessaire
    # d["MATRICULE"] = d["MATRICULE"].str.upper()

# --- 3) Déterminer les colonnes en conflit (même nom des deux côtés), hors clés
keys = {"SOURCE_FICHIER", "MATRICULE"}
overlap = [c for c in dfL.columns if c in dfR.columns and c not in keys]

# --- 4) (Optionnel) Ne rapatrier que l’utile côté dfR (réduit la mémoire)
#    -> on garde les clés + (colonnes R non présentes à gauche) + (colonnes en conflit pour coalesce)
cols_R_keep = ["SOURCE_FICHIER", "MATRICULE"]
cols_R_keep += [c for c in dfR.columns if c not in dfL.columns and c not in keys]
cols_R_keep += [c for c in overlap if c in dfR.columns]
dfR = dfR.loc[:, sorted(set(cols_R_keep), key=cols_R_keep.index)]

# --- 5) Merge (LEFT) avec suffixes pour distinguer les colonnes en conflit
merged = dfL.merge(
    dfR,
    on=["SOURCE_FICHIER","MATRICULE"],
    how="left",
    suffixes=("", "_b1"),
    indicator=True
)

# --- 6) Résoudre les colonnes dupliquées : garder la gauche et compléter par la droite si NaN
for col in overlap:
    col_right = f"{col}_b1"
    if col_right in merged.columns:
        # complète la colonne gauche si elle est vide
        merged[col] = merged[col].astype("object").where(merged[col].notna(), merged[col_right])
        merged.drop(columns=[col_right], inplace=True)



NameError: name 'panel_solde_df' is not defined

## Bilan de la fusion

In [ ]:
print("Bilan _merge:")
print(merged["_merge"].value_counts(dropna=False))
if len(merged) > len(dfL):
    print(f"⚠️ many-to-many détecté : lignes left={len(dfL):,} → après merge={len(merged):,}")

# Résultat final
panel_solde_df_final = merged
